In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import root_mean_squared_error, mean_absolute_error

# =====================================================================
# 1. CREATE MOCK DATASET (Simulating your data structure)
# =====================================================================
# This mimics the training data sample you provided
train_data = pd.read_csv("data_preview.csv")

# This mimics the unlabeled test dataset Kaggle gives you
test_data = pd.DataFrame({
    'ENCOUNTER': [105256127, 105256137],
    'PATIENT_NUM': [9922060135, 9922071234],
    'DOCTOR': ['202927', '306822'],
    'DEPARTMENT': ['GENERAL MED', 'HEART']
    # 'ADMIT_LOS' is missing here, we must predict it!
})

# =====================================================================
# 2. SEPARATE FEATURES AND TARGET
# =====================================================================
# We drop ENCOUNTER and PATIENT_NUM from training features because raw IDs confuse models
X = train_data[['DOCTOR', 'DEPARTMENT']]
y = train_data['ADMIT_LOS']

# =====================================================================
# 3. PREPROCESSING & MODEL PIPELINE
# =====================================================================
# We convert text columns ('DOCTOR', 'DEPARTMENT') into numbers using One-Hot Encoding
categorical_features = ['DOCTOR', 'DEPARTMENT']
categorical_transformer = OneHotEncoder(handle_unknown='ignore')

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', categorical_transformer, categorical_features)
    ]
)

# Bundle preprocessing and the Random Forest model together
model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(n_estimators=100, random_state=42))
])

# =====================================================================
# 4. TRAIN AND EVALUATE (Validation Split)
# =====================================================================
# Split training data to see how well the model is performing locally
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.25, random_state=42)

# Train the model
model_pipeline.fit(X_train, y_train)

# Calculate local performance metrics
val_predictions = model_pipeline.predict(X_val)
rmse = root_mean_squared_error(y_val, val_predictions)
mae = mean_absolute_error(y_val, val_predictions)

print(f"--- Local Validation Scores ---")
print(f"RMSE (Leaderboard Metric): {rmse:.4f}")
print(f"MAE (Avg Days Off): {mae:.4f}\n")

# =====================================================================
# 5. PREDICT ON UNSEEN TEST DATA & SAVE SUBMISSION
# =====================================================================
# Retrain on the full training set before final prediction for max accuracy
model_pipeline.fit(X, y)

# Predict the length of stay for the test set
test_features = test_data[['DOCTOR', 'DEPARTMENT']]
test_data['ADMIT_LOS'] = model_pipeline.predict(test_features)

# Format the final output to match Kaggle instructions perfectly
submission = test_data[['ENCOUNTER', 'ADMIT_LOS']].rename(columns={'ENCOUNTER': 'ENCOUNTER_KEY'})

# Save to a CSV file
submission.to_csv('submission.csv', index=False)
print("--- Final Submission File Preview ---")
print(submission.to_string(index=False))

--- Local Validation Scores ---
RMSE (Leaderboard Metric): 4.5314
MAE (Avg Days Off): 3.4267

--- Final Submission File Preview ---
 ENCOUNTER_KEY  ADMIT_LOS
     105256127       2.93
     105256137       3.21
